# Course 1, Week 3 — Data Management in PyTorch

- [DeepLearning.AI platform](https://learn.deeplearning.ai/specializations/pytorch-for-deep-learning-professional-certificate/lesson)
- [Week notes](README.md)
- [GitHub issue #3](https://github.com/majorgilles/pytorch_for_deep_learning/issues/3)

**Focus:** Represent datasets and feed repeatable batches through Dataset and DataLoader.

## Video guide

| Video | Covered notebook section |
|---|---|
| [Introduction to Data Pipelines](https://learn.deeplearning.ai/specializations/pytorch-for-deep-learning-professional-certificate/lesson/bvvjkv/introduction-to-data-pipelines) | Introduction to data pipelines |
| Accessing messy data with a custom `Dataset` | Solving data-access problems with a custom `Dataset` |
| Solving image-quality problems with transforms | Solving image-quality problems with transforms |

> The remaining video titles will receive direct DeepLearning.AI links when their lesson URLs are added.


In [1]:
import os
import tarfile
from typing import cast

import numpy as np
import requests
import torch
from numpy import typing as npt
from scipy.io import loadmat
from PIL import Image
from torch.utils.data import Dataset
from tqdm import tqdm

In [2]:
def download_dataset() -> None:
    """
    Downloads and extracts a dataset from remote URLs if not already present locally.

    This function first checks for the existence of the dataset files in a specific
    directory. If the files are not found, it proceeds to download them from
    pre-defined URLs, and then extracts the contents.
    """
    # Define the directory to store the dataset.
    data_dir: str = "flower_data"

    # Define paths for key files and folders.
    image_folder_path: str = os.path.join(data_dir, "jpg")
    labels_file_path: str = os.path.join(data_dir, "imagelabels.mat")
    tgz_path: str = os.path.join(data_dir, "102flowers.tgz")

    # Check if the primary data folder and a key label file already exist.
    if os.path.exists(image_folder_path) and os.path.exists(labels_file_path):
        # Inform the user that the dataset is already available locally.
        print(f"Dataset already exists. Loading locally from '{data_dir}'.")
        # Exit the function since no download is needed.
        return

    # Inform the user that the dataset is not found and the download will start.
    print("Dataset not found locally. Downloading...")

    # Define the URLs for the image archive and the labels file.
    image_url: str = "https://www.robots.ox.ac.uk/~vgg/data/flowers/102/102flowers.tgz"
    labels_url: str = "https://www.robots.ox.ac.uk/~vgg/data/flowers/102/imagelabels.mat"

    # Create the target directory for the dataset, if it doesn't already exist.
    os.makedirs(data_dir, exist_ok=True)

    # Announce the start of the image download process.
    print("Downloading images...")
    # Send an HTTP GET request to the image URL, enabling streaming for large files.
    response: requests.Response = requests.get(image_url, stream=True, timeout=60)
    response.raise_for_status()
    # Get the total size of the file from the response headers for the progress bar.
    total_size: int = int(response.headers.get("content-length", 0))

    # Open a local file in binary write mode to save the downloaded archive.
    with open(tgz_path, "wb") as file:
        # Iterate over the response content in chunks with a progress bar.
        file.writelines(tqdm(
            # Define the chunk size for iterating over the content.
            response.iter_content(chunk_size=1024),
            # Set the total for the progress bar based on the file size in kilobytes.
            total=total_size // 1024,
        ))

    # Announce the start of the file extraction process.
    print("Extracting files...")
    # Open the downloaded tar.gz archive in read mode.
    with tarfile.open(tgz_path, "r:gz") as tar:
        # Extract all contents of the archive into the target directory.
        tar.extractall(data_dir, filter="data")

    # Announce the start of the labels download process.
    print("Downloading labels...")
    # Send an HTTP GET request to the labels URL.
    response = requests.get(labels_url, timeout=60)
    response.raise_for_status()
    # Open a local file in binary write mode to save the labels.
    with open(labels_file_path, "wb") as file:
        # Write the entire content of the response to the file.
        file.write(response.content)

    # Inform the user that the download and extraction are complete.
    print(f"Dataset downloaded and extracted to '{data_dir}'.")

    # create labels_description.txt
    labels_description: list[str] = [
        "pink primrose",
        "hard-leaved pocket orchid",
        "canterbury bells",
        "sweet pea",
        "english marigold",
        "tiger lily",
        "moon orchid",
        "bird of paradise",
        "monkshood",
        "globe thistle",
        "snapdragon",
        "colt's foot",
        "king protea",
        "spear thistle",
        "yellow iris",
        "globe-flower",
        "purple coneflower",
        "peruvian lily",
        "balloon flower",
        "giant white arum lily",
        "fire lily",
        "pincushion flower",
        "fritillary",
        "red ginger",
        "grape hyacinth",
        "corn poppy",
        "prince of wales feathers",
        "stemless gentian",
        "artichoke",
        "sweet william",
        "carnation",
        "garden phlox",
        "love in the mist",
        "mexican aster",
        "alpine sea holly",
        "ruby-lipped cattleya",
        "cape flower",
        "great masterwort",
        "siam tulip",
        "lenten rose",
        "barbeton daisy",
        "daffodil",
        "sword lily",
        "poinsettia",
        "bolero deep blue",
        "wallflower",
        "marigold",
        "buttercup",
        "oxeye daisy",
        "common dandelion",
        "petunia",
        "wild pansy",
        "primula",
        "sunflower",
        "pelargonium",
        "bishop of llandaff",
        "gaura",
        "geranium",
        "orange dahlia",
        "pink-yellow dahlia?",
        "cautleya spicata",
        "japanese anemone",
        "black-eyed susan",
        "silverbush",
        "californian poppy",
        "osteospermum",
        "spring crocus",
        "bearded iris",
        "windflower",
        "tree poppy",
        "gazania",
        "azalea",
        "water lily",
        "rose",
        "thorn apple",
        "morning glory",
        "passion flower",
        "lotus",
        "toad lily",
        "anthurium",
        "frangipani",
        "clematis",
        "hibiscus",
        "columbine",
        "desert-rose",
        "tree mallow",
        "magnolia",
        "cyclamen ",
        "watercress",
        "canna lily",
        "hippeastrum ",
        "bee balm",
        "ball moss",
        "foxglove",
        "bougainvillea",
        "camellia",
        "mallow",
        "mexican petunia",
        "bromelia",
        "blanket flower",
        "trumpet creeper",
        "blackberry lily",
    ]

    with open(
        os.path.join(data_dir, "labels_description.txt"), "w", encoding="utf-8"
    ) as file:
        file.writelines(f"{label}\n" for label in labels_description)

In [3]:
download_dataset()

Dataset already exists. Loading locally from 'flower_data'.


In [4]:
class OxfordFlowersDataset(Dataset[tuple[Image.Image, int]]):
    """Lazily pair Oxford Flowers images with zero-based class labels."""

    def __init__(self, root_dir: str) -> None:
        self.root_dir: str = root_dir
        self.img_dir: str = os.path.join(root_dir, "jpg")

        # Load lightweight metadata now; image pixels remain on disk.
        raw_labels: npt.NDArray[np.int64] = cast(
            npt.NDArray[np.int64],
            np.asarray(
                loadmat(os.path.join(root_dir, "imagelabels.mat"))["labels"][0],
                dtype=np.int64,
            ),
        )
        self.labels: npt.NDArray[np.int64] = raw_labels - 1

    def __len__(self) -> int:
        """Return the number of image-label pairs."""
        return len(self.labels)

    def __getitem__(self, index: int) -> tuple[Image.Image, int]:
        """Load one RGB image and its zero-based label by dataset index."""
        # Dataset indices start at 0, while Oxford filenames start at 1.
        image_name: str = f"image_{index + 1:05d}.jpg"
        image_path: str = os.path.join(self.img_dir, image_name)

        # Materialize an RGB copy so the source file can close safely.
        with Image.open(image_path) as source_image:
            image: Image.Image = source_image.convert("RGB")

        label: int = int(self.labels[index])
        return image, label

In [5]:
dataset: OxfordFlowersDataset = OxfordFlowersDataset(root_dir="./flower_data")
print(f"Total samples: {len(dataset)}")
img, label = dataset[0]
img, label

Total samples: 8189


(<PIL.Image.Image image mode=RGB size=591x500>, 76)